# 12. Capstone: DevSecOps Pipeline

## Background

This capstone integrates every tool from Series 34 into a unified DevSecOps pipeline. In production, this pipeline runs as a CI/CD workflow on every pull request — each 'gate' is a mandatory check that blocks deployment if it fails.

The architecture follows the OWASP DevSecOps Guideline: security controls embedded at every phase of the SDLC rather than bolted on at the end. Teams at Google, Netflix, and Shopify run variations of this pipeline at scale, processing thousands of pull requests daily with automated security gates and human escalation only for high-severity findings.

## What You'll Learn

- Composing SAST, DAST, SBOM, supply chain, secrets, and container checks into one pipeline
- Designing a gate policy: which failures block vs warn
- Security posture scoring: aggregate metric across all controls
- Pipeline reporting: developer-friendly output + SARIF export for GitHub Security tab

## Why This Matters

Security without automation is a policy document. This pipeline turns every concept from Series 34 into code that runs on every commit — making security as routine and invisible as formatting checks. For LLM applications shipping rapidly, automated security gates are the only way to maintain security posture without slowing down iteration.


## Pipeline Gate Definitions

In [ ]:
from dataclasses import dataclass, field
from typing import List, Dict, Callable, Optional, Any
from enum import Enum
import json
import time

class GateResult(Enum):
    PASS = 'PASS'
    WARN = 'WARN'
    FAIL = 'FAIL'
    SKIP = 'SKIP'

@dataclass
class GateOutput:
    gate_name: str
    result: GateResult
    findings_count: int
    critical_count: int
    high_count: int
    duration_ms: int
    details: List[str] = field(default_factory=list)

class DevSecOpsPipeline:
    def __init__(self, project_name: str, commit_sha: str):
        self.project_name = project_name
        self.commit_sha = commit_sha
        self.gates: List[Dict] = []
        self.outputs: List[GateOutput] = []

    def add_gate(self, name: str, fn: Callable, blocking: bool = True,
                 fail_on_critical: bool = True, fail_on_high: bool = False):
        self.gates.append({
            'name': name, 'fn': fn, 'blocking': blocking,
            'fail_on_critical': fail_on_critical, 'fail_on_high': fail_on_high
        })

    def run(self, context: Dict) -> bool:
        print(f'=== DevSecOps Pipeline: {self.project_name} ===')
        print(f'Commit: {self.commit_sha[:8]}')
        print(f'Gates:  {len(self.gates)}')
        print()

        pipeline_passed = True

        for gate_def in self.gates:
            start = time.time()
            try:
                output = gate_def['fn'](context)
            except Exception as e:
                output = GateOutput(
                    gate_name=gate_def['name'], result=GateResult.FAIL,
                    findings_count=0, critical_count=0, high_count=0,
                    duration_ms=0, details=[f'Gate error: {e}']
                )
            output.duration_ms = int((time.time() - start) * 1000)

            # Override result based on gate config
            if gate_def['fail_on_critical'] and output.critical_count > 0:
                output.result = GateResult.FAIL
            elif gate_def['fail_on_high'] and output.high_count > 0:
                output.result = GateResult.FAIL

            self.outputs.append(output)

            icon = {'PASS': '[PASS]', 'WARN': '[WARN]', 'FAIL': '[FAIL]', 'SKIP': '[SKIP]'}[output.result.value]
            blocking_note = '' if gate_def['blocking'] else ' (non-blocking)'
            print(f'{icon} {output.gate_name:35s} {output.findings_count:3d} findings  {output.duration_ms}ms{blocking_note}')

            if gate_def['blocking'] and output.result == GateResult.FAIL:
                pipeline_passed = False

        return pipeline_passed

    def score(self) -> float:
        '''Security posture score 0-100: 100 = all gates pass.'''
        if not self.outputs: return 0.0
        passed = sum(1 for o in self.outputs if o.result == GateResult.PASS)
        return round(100 * passed / len(self.outputs), 1)

print('Pipeline framework loaded')


## Gate Implementations

In [ ]:
import re
import random

# --- Gate functions (wiring back to earlier notebooks' logic) ---

def sast_gate(context: Dict) -> GateOutput:
    code = context.get('source_code', '')
    PATTERNS = [
        (r'eval\s*\(', 'HIGH', 'eval() usage'),
        (r'os\.system\s*\(', 'HIGH', 'os.system() usage'),
        (r'pickle\.loads?\s*\(', 'HIGH', 'pickle deserialization'),
        (r'torch\.load\s*\(', 'CRITICAL', 'unsafe torch.load()'),
        (r'subprocess.*shell\s*=\s*True', 'HIGH', 'shell=True in subprocess'),
        (r'f".*SELECT.*{', 'CRITICAL', 'f-string SQL construction'),
    ]
    findings = []
    for pattern, severity, msg in PATTERNS:
        if re.search(pattern, code):
            findings.append(f'{severity}: {msg}')
    critical = sum(1 for f in findings if f.startswith('CRITICAL'))
    high = sum(1 for f in findings if f.startswith('HIGH'))
    result = GateResult.FAIL if critical > 0 else (GateResult.WARN if high > 0 else GateResult.PASS)
    return GateOutput('SAST', result, len(findings), critical, high, 0, findings)

def secrets_gate(context: Dict) -> GateOutput:
    import math, collections
    def entropy(s):
        if not s: return 0
        freq = collections.Counter(s)
        n = len(s)
        return -sum((c/n)*math.log2(c/n) for c in freq.values())

    code = context.get('source_code', '')
    SECRET_RE = re.compile(r'(sk-[a-zA-Z0-9]{20,}|AKIA[0-9A-Z]{16}|-----BEGIN.*PRIVATE)')
    findings = []
    for line in code.splitlines():
        if SECRET_RE.search(line):
            findings.append(f'CRITICAL: potential secret on line: {line.strip()[:60]}')
        for token in re.split(r'[\s=:"]+', line):
            if len(token) >= 24 and entropy(token) > 4.8:
                findings.append(f'HIGH: high-entropy string: {token[:20]}...')
    critical = sum(1 for f in findings if 'CRITICAL' in f)
    high = sum(1 for f in findings if f.startswith('HIGH'))
    result = GateResult.FAIL if critical > 0 else (GateResult.WARN if high > 0 else GateResult.PASS)
    return GateOutput('Secrets Detection', result, len(findings), critical, high, 0, findings)

def dependency_gate(context: Dict) -> GateOutput:
    deps = context.get('dependencies', {})
    KNOWN_VULNS = {
        'langchain': ('0.0.247', 'CRITICAL'),
        'gradio': ('3.32.0', 'HIGH'),
        'aiohttp': ('3.8.5', 'HIGH'),
    }
    findings = []
    for pkg, version in deps.items():
        if pkg in KNOWN_VULNS:
            fix_ver, severity = KNOWN_VULNS[pkg]
            installed = tuple(int(x) for x in version.split('.'))
            threshold = tuple(int(x) for x in fix_ver.split('.'))
            if installed < threshold:
                findings.append(f'{severity}: {pkg}=={version} vulnerable (fix: >={fix_ver})')
    critical = sum(1 for f in findings if f.startswith('CRITICAL'))
    high = sum(1 for f in findings if f.startswith('HIGH'))
    result = GateResult.FAIL if critical > 0 else (GateResult.WARN if high > 0 else GateResult.PASS)
    return GateOutput('Dependency Scan', result, len(findings), critical, high, 0, findings)

def dockerfile_gate(context: Dict) -> GateOutput:
    dockerfile = context.get('dockerfile', '')
    findings = []
    if ':latest' in dockerfile: findings.append('HIGH: FROM with :latest tag')
    if 'USER root' in dockerfile or 'USER 0' in dockerfile: findings.append('HIGH: running as root')
    if 'OPENAI_API_KEY' in dockerfile or 'API_KEY' in dockerfile: findings.append('CRITICAL: secret in Dockerfile')
    if not any(line.strip().startswith('USER') and 'root' not in line for line in dockerfile.splitlines()):
        findings.append('HIGH: no non-root USER instruction')
    critical = sum(1 for f in findings if f.startswith('CRITICAL'))
    high = sum(1 for f in findings if f.startswith('HIGH'))
    result = GateResult.FAIL if critical > 0 else (GateResult.WARN if high > 0 else GateResult.PASS)
    return GateOutput('Container/Dockerfile', result, len(findings), critical, high, 0, findings)

def sbom_license_gate(context: Dict) -> GateOutput:
    licenses = context.get('licenses', [])
    BLOCKED = ['GPL-3.0', 'AGPL-3.0']
    findings = [f'HIGH: {lic} license — copyleft risk' for lic in licenses if lic in BLOCKED]
    high = len(findings)
    result = GateResult.WARN if high > 0 else GateResult.PASS
    return GateOutput('SBOM/License Check', result, len(findings), 0, high, 0, findings)

print('Gate functions defined')


## Full Pipeline Run

In [ ]:
# Build pipeline
pipeline = DevSecOpsPipeline('llm-chat-app', 'a1b2c3d4e5f6')
pipeline.add_gate('SAST', sast_gate, blocking=True, fail_on_critical=True)
pipeline.add_gate('Secrets Detection', secrets_gate, blocking=True, fail_on_critical=True)
pipeline.add_gate('Dependency Scan', dependency_gate, blocking=True, fail_on_critical=True)
pipeline.add_gate('Container/Dockerfile', dockerfile_gate, blocking=True, fail_on_critical=True)
pipeline.add_gate('SBOM/License', sbom_license_gate, blocking=False)

# Simulate CI context
context = {
    'source_code': '''
import torch
model = torch.load('model.pt')  # unsafe deserialization
user_input = request.json['q']
results = db.execute(f"SELECT * FROM logs WHERE user='{user_input}'")  # SQLi
''',
    'dependencies': {
        'langchain': '0.0.200',  # < 0.0.247
        'fastapi': '0.104.0',
        'aiohttp': '3.8.3',     # < 3.8.5
    },
    'dockerfile': '''
FROM python:latest
ENV OPENAI_API_KEY=sk-test123
CMD ["python", "app.py"]
''',
    'licenses': ['MIT', 'Apache-2.0', 'AGPL-3.0'],
}

passed = pipeline.run(context)
score = pipeline.score()

print(f'\nSecurity Posture Score: {score}/100')
print(f'Pipeline Result: {"PASS" if passed else "FAIL"}')

# Show detailed failures
failures = [o for o in pipeline.outputs if o.result == GateResult.FAIL]
if failures:
    print(f'\nBlocking failures:')
    for gate in failures:
        for detail in gate.details[:3]:
            print(f'  {gate.gate_name}: {detail}')


## Security Posture Improvement Tracking

In [ ]:
# After fixing the issues, re-run with clean code
clean_context = {
    'source_code': '''
import torch
model = torch.load('model.pt', weights_only=True)  # safe
user_input = request.json['q']
results = db.execute("SELECT * FROM logs WHERE user = %s", (user_input,))  # parameterized
''',
    'dependencies': {
        'langchain': '0.0.260',  # fixed
        'fastapi': '0.104.0',
        'aiohttp': '3.9.0',     # fixed
    },
    'dockerfile': '''
FROM python:3.11-slim
RUN useradd -r appuser
USER appuser
CMD ["python", "app.py"]
''',
    'licenses': ['MIT', 'Apache-2.0'],  # removed AGPL
}

clean_pipeline = DevSecOpsPipeline('llm-chat-app-fixed', 'f9e8d7c6b5a4')
clean_pipeline.add_gate('SAST', sast_gate, blocking=True, fail_on_critical=True)
clean_pipeline.add_gate('Secrets Detection', secrets_gate, blocking=True, fail_on_critical=True)
clean_pipeline.add_gate('Dependency Scan', dependency_gate, blocking=True, fail_on_critical=True)
clean_pipeline.add_gate('Container/Dockerfile', dockerfile_gate, blocking=True, fail_on_critical=True)
clean_pipeline.add_gate('SBOM/License', sbom_license_gate, blocking=False)

print('=== After Remediation ===')
clean_passed = clean_pipeline.run(clean_context)
clean_score = clean_pipeline.score()

print(f'\nSecurity Posture Score: {clean_score}/100 (was {score}/100)')
print(f'Pipeline Result: {"PASS" if clean_passed else "FAIL"}')

print('\n=== Series 34 Complete: Secure Software Development ===')
print('Topics covered:')
topics = [
    '01. Secure SDLC & DevSecOps',
    '02. Threat Modelling (STRIDE, Attack Trees)',
    '03. SAST for General Code',
    '04. SAST for ML Pipelines',
    '05. DAST Automation',
    '06. Dependency Scanning & CVE Detection',
    '07. SBOM Generation & Analysis',
    '08. Supply Chain Security (Sigstore, SLSA)',
    '09. Secrets Detection & Pre-commit Hooks',
    '10. Container Security & Image Hardening',
    '11. LLM-Assisted Secure Code Review',
    '12. DevSecOps Pipeline Capstone',
]
for t in topics:
    print(f'  {t}')
